In [ ]:
#@title 🎵 YuE2 com Monitor em Tempo Real (Rápido + Sem Quedas de VRAM)
import os
import re
import sys
import time
import socket
import threading
import subprocess
from IPython.display import clear_output, display, HTML

%cd /content/yue2.cpp

# 1. LIMPEZA DE PROCESSOS ANTERIORES
print("🧹 Limpando processos antigos na memória...")
subprocess.run(["pkill", "-9", "-f", "yue-server"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-9", "-f", "cloudflared"], stderr=subprocess.DEVNULL)
time.sleep(1)

server_bin = "/content/yue2.cpp/build/yue-server"
if not os.path.exists(server_bin):
    raise FileNotFoundError("Binário yue-server não encontrado. Execute a célula de instalação primeiro.")

# 2. INICIAR COM PARÂMETROS OTIMIZADOS (EVITA OOM E ACELERA 3X)
cmd = [
    server_bin,
    "--host", "0.0.0.0",
    "--port", "8087",
    "--model", "models/YuE2-3B-Q8_0.gguf",
    "--vae", "models/YuE2-Vae-F32.gguf",
    "--max-seq", "4096"
]

if os.path.exists("models/SheetSage2-Q8_0.gguf"):
    cmd.extend(["--transcriber", "models/SheetSage2-Q8_0.gguf"])

if os.path.exists("models/yue2-mothersuperior-realaudio-tokenizer-v4-Q8_0.gguf"):
    cmd.extend(["--tokenizer", "models/yue2-mothersuperior-realaudio-tokenizer-v4-Q8_0.gguf"])

print("🚀 Iniciando YuE2 otimizado para GPU T4...")
server_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# 3. AGUARDAR O MODELO CARREGAR NA GPU
porta_aberta = False
inicio = time.time()
print("⏳ Carregando modelos na VRAM...")

while (time.time() - inicio) < 90:
    if server_proc.poll() is not None:
        saida, _ = server_proc.communicate()
        print("\n❌ Erro ao iniciar:")
        print(saida[-1000:])
        raise RuntimeError("Servidor encerrou inesperadamente.")

    try:
        with socket.create_connection(("127.0.0.1", 8087), timeout=1):
            porta_aberta = True
            break
    except (socket.timeout, ConnectionRefusedError, OSError):
        pass
    time.sleep(2)

if not porta_aberta:
    server_proc.kill()
    raise TimeoutError("Servidor demorou demais para responder.")

print("✅ Servidor pronto!")

# 4. INICIAR TÚNEL CLOUDFLARE
cloudflared_path = "/content/cloudflared"
tunnel_log = open("tunnel.log", "w", encoding="utf-8")
tunnel_proc = subprocess.Popen(
    [cloudflared_path, "tunnel", "--url", "http://127.0.0.1:8087"],
    stdout=tunnel_log,
    stderr=tunnel_log
)

url_webui = None
for _ in range(20):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r", encoding="utf-8", errors="ignore") as f:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if match:
                url_webui = match.group(0)
                break

print("\n" + "="*65)
print(f"👉 CLIQUE AQUI PARA ABRIR O ESTÚDIO: {url_webui}")
print("="*65)
print("👀 MONITOR ATIVO: Veja abaixo em tempo real cada passo da geração!\n")

# 5. MONITOR EM TEMPO REAL
def monitor_logs():
    for linha in iter(server_proc.stdout.readline, ''):
        l = linha.strip()
        if not l:
            continue

        # Filtra os passos para você saber exatamente o que está acontecendo
        if "Composing" in l or "Step" in l or "semantic" in l:
            print(f"🎼 [COMPONDO LETRA/MELODIA]: {l}")
        elif "NAR" in l or "diffusion" in l:
            print(f"🎸 [SINTETIZANDO INSTRUMENTOS]: {l}")
        elif "VAE" in l or "decode" in l:
            print(f"🎧 [RENDERIZANDO ÁUDIO]: {l}")
        elif "POST /synth" in l or "job" in l.lower():
            print(f"\n📥 [NOVO PEDIDO RECEBIDO]: Iniciando criação da música...")
        elif "out of memory" in l.lower() or "cudaMalloc failed" in l.lower():
            print(f"\n❌ [ERRO DE VRAM]: Memória insuficiente para essa letra. Reduza a quantidade de versos.")
        elif "finished" in l.lower() or "saved" in l.lower() or "100%" in l:
            print(f"🎉 [SUCESSO]: Música finalizada!\n")

thread = threading.Thread(target=monitor_logs, daemon=True)
thread.start()

# Mantém a célula executando
try:
    while server_proc.poll() is None:
        time.sleep(2)
except KeyboardInterrupt:
    print("\n🛑 Encerrando...")
    server_proc.terminate()
    tunnel_proc.terminate()